<a href="https://colab.research.google.com/github/Muhammad-Arslan6621/Heart-Disease-Prediction-ML/blob/main/Heart_Disease_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

# 1. Set your credentials (from your screenshot)
os.environ['KAGGLE_USERNAME'] = "arslankahloon16"
os.environ['KAGGLE_KEY'] = "KGAT_9f122000cf045b9d865aec99e2e604f9"

# 2. Install kagglehub (the latest tool for Kaggle)
!pip install kagglehub -q

import kagglehub

# 3. Download the Heart Disease dataset
# (Using a common heart disease dataset as an example)
path = kagglehub.dataset_download("juhibhojani/heart-diease")

print("Data downloaded to:", path)

100%|██████████| 55.3M/55.3M [00:00<00:00, 124MB/s]

Extracting files...


Data downloaded to: /root/.cache/kagglehub/datasets/juhibhojani/heart-diease/versions/1


In [5]:
# Filter for only "All heart disease" records
heart_disease = heart_disease[heart_disease['Topic'] == 'All heart disease']

# Filter for only "County" or "State" level if needed
# heart_disease = heart_disease[heart_disease['GeographicLevel'] == 'County']

In [6]:
# Check how many missing values are in each column
print(heart_disease.isnull().sum())

# Option A: Drop rows where the actual data value is missing
heart_disease = heart_disease.dropna(subset=['Data_Value'])

# Option B: Fill missing numbers with the average (Mean)
# heart_disease['Data_Value'] = heart_disease['Data_Value'].fillna(heart_disease['Data_Value'].mean())

Year                               0
LocationAbbr                       0
LocationDesc                       0
GeographicLevel                    0
DataSource                         0
Class                              0
Topic                              0
Data_Value                    473095
Data_Value_Unit                    0
Data_Value_Type                    0
Data_Value_Footnote_Symbol    680953
Data_Value_Footnote           680953
Confidence_limit_Low          473095
Confidence_limit_High         473095
StratificationCategory1            0
Stratification1                    0
StratificationCategory2            0
Stratification2                    0
StratificationCategory3            0
Stratification3                    0
LocationID                         0
dtype: int64


In [7]:
# Keep only the columns that actually matter for analysis
columns_to_keep = ['Year', 'LocationDesc', 'GeographicLevel', 'Data_Value', 'Data_Value_Unit', 'Stratification1', 'Stratification2']
heart_disease = heart_disease[columns_to_keep]

# See your clean data
heart_disease.head()

,Year,LocationDesc,GeographicLevel,Data_Value,Data_Value_Unit,Stratification1,Stratification2
108,2017,Autauga,County,128.7,"per 100,000",Ages 35-64 years,Overall
109,2016,Autauga,County,128.1,"per 100,000",Ages 35-64 years,Overall
110,2019,Autauga,County,122.6,"per 100,000",Ages 35-64 years,Overall
111,2006,Autauga,County,164.0,"per 100,000",Ages 35-64 years,Overall
112,2003,Autauga,County,159.6,"per 100,000",Ages 35-64 years,Overall


In [8]:
heart_disease

,Year,LocationDesc,GeographicLevel,Data_Value,Data_Value_Unit,Stratification1,Stratification2
108,2017,Autauga,County,128.7,"per 100,000",Ages 35-64 years,Overall
109,2016,Autauga,County,128.1,"per 100,000",Ages 35-64 years,Overall
110,2019,Autauga,County,122.6,"per 100,000",Ages 35-64 years,Overall
111,2006,Autauga,County,164.0,"per 100,000",Ages 35-64 years,Overall
112,2003,Autauga,County,159.6,"per 100,000",Ages 35-64 years,Overall
...,...,...,...,...,...,...,...
5770181,1999 - 2010,Weston,County,-27.5,%,Ages 65 years and older,Overall
5770190,2010 - 2019,Weston,County,-4.1,%,Ages 65 years and older,Overall
5770191,1999 - 2010,Weston,County,-22.3,%,Ages 65 years and older,Overall
5770230,1999 - 2010,Weston,County,-24.8,%,Ages 65 years and older,White


In [9]:
# 1. Keep only rows that use "per 100,000" (Standard Rate)
heart_disease = heart_disease[heart_disease['Data_Value_Unit'] == 'per 100,000']

# 2. Keep only rows with single years (not ranges like 1999-2010)
# This removes the negative trend data automatically
heart_disease = heart_disease[heart_disease['Year'].str.len() == 4]

# 3. Convert Year to an integer for the model
heart_disease['Year'] = heart_disease['Year'].astype(int)

print(f"Remaining rows after cleaning: {len(heart_disease)}")

Remaining rows after cleaning: 621744


In [10]:
# Use "One-Hot Encoding" to turn text categories into columns of 0s and 1s
heart_disease_final = pd.get_dummies(heart_disease, columns=['Stratification1', 'Stratification2'])

# Drop columns that are no longer needed
heart_disease_final = heart_disease_final.drop(['LocationDesc', 'GeographicLevel', 'Data_Value_Unit'], axis=1)

heart_disease_final.head()

,Year,Data_Value,Stratification1_Ages 35-64 years,Stratification1_Ages 65 years and older,Stratification2_American Indian/Alaska Native,Stratification2_Asian/Pacific Islander,Stratification2_Black (Non-Hispanic),Stratification2_Hispanic,Stratification2_Overall,Stratification2_White
108,2017,128.7,True,False,False,False,False,False,True,False
109,2016,128.1,True,False,False,False,False,False,True,False
110,2019,122.6,True,False,False,False,False,False,True,False
111,2006,164.0,True,False,False,False,False,False,True,False
112,2003,159.6,True,False,False,False,False,False,True,False


In [11]:
# Check for any remaining missing values
print("Missing values per column:")
print(heart_disease_final.isnull().sum())

# Check the range of your target value
print("\nData Value Statistics:")
print(heart_disease_final['Data_Value'].describe())

Missing values per column:
Year                                             0
Data_Value                                       0
Stratification1_Ages 35-64 years                 0
Stratification1_Ages 65 years and older          0
Stratification2_American Indian/Alaska Native    0
Stratification2_Asian/Pacific Islander           0
Stratification2_Black (Non-Hispanic)             0
Stratification2_Hispanic                         0
Stratification2_Overall                          0
Stratification2_White                            0
dtype: int64

Data Value Statistics:
count    621744.000000
mean        696.120632
std         689.009303
min          12.100000
25%         100.400000
50%         222.800000
75%        1285.600000
max        3993.400000
Name: Data_Value, dtype: float64


In [12]:
heart_disease

,Year,LocationDesc,GeographicLevel,Data_Value,Data_Value_Unit,Stratification1,Stratification2
108,2017,Autauga,County,128.7,"per 100,000",Ages 35-64 years,Overall
109,2016,Autauga,County,128.1,"per 100,000",Ages 35-64 years,Overall
110,2019,Autauga,County,122.6,"per 100,000",Ages 35-64 years,Overall
111,2006,Autauga,County,164.0,"per 100,000",Ages 35-64 years,Overall
112,2003,Autauga,County,159.6,"per 100,000",Ages 35-64 years,Overall
...,...,...,...,...,...,...,...
5268410,2015,Weston,County,1003.4,"per 100,000",Ages 65 years and older,White
5268411,2003,Weston,County,1349.5,"per 100,000",Ages 65 years and older,White
5268412,2000,Weston,County,1376.7,"per 100,000",Ages 65 years and older,White
5268413,2001,Weston,County,1369.3,"per 100,000",Ages 65 years and older,White


In [17]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Create a binary target: 1 if heart disease rate is above median, 0 if below
median_rate = heart_disease_final['Data_Value'].median()
heart_disease_final['High_Risk'] = (heart_disease_final['Data_Value'] > median_rate).astype(int)

# 2. Define Features (X) and Target (y)
X = heart_disease_final.drop(['Data_Value', 'High_Risk'], axis=1)
y = heart_disease_final['High_Risk']

# 3. Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [18]:
from sklearn.ensemble import RandomForestClassifier

# We instantiate the model (create the object)
clf = RandomForestClassifier()

In [19]:
# Fitting the model (Training)
clf.fit(X_train, y_train)

# Making predictions (Testing)
y_preds = clf.predict(X_test)

In [20]:
from sklearn.metrics import classification_report, accuracy_score

print(f"Model Accuracy: {accuracy_score(y_test, y_preds) * 100:.2f}%")
print("\nDetailed Report:")
print(classification_report(y_test, y_preds))

Model Accuracy: 96.26%

Detailed Report:
              precision    recall  f1-score   support

           0       0.94      0.99      0.96     62050
           1       0.99      0.93      0.96     62299

    accuracy                           0.96    124349
   macro avg       0.96      0.96      0.96    124349
weighted avg       0.96      0.96      0.96    124349



In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# 1. Define the 'Hyperparameter Grid' (the settings we want to test)
grid = {
    "n_estimators": [100, 200, 500],
    "max_depth": [None, 10, 20],
    "max_features": ["sqrt", "log2"],
    "min_samples_split": [2, 6]
}

# 2. Setup the search
# n_iter=5 means it will try 5 random combinations from the grid
rs_clf = RandomizedSearchCV(estimator=clf,
                            param_distributions=grid,
                            n_iter=5,
                            cv=5,
                            verbose=2)

# 3. Fit the tuned version (This might take a minute)
rs_clf.fit(X_train, y_train)

# 4. See the best settings it found
print("Best Hyperparameters:", rs_clf.best_params_)

Fitting 5 folds for each of 5 candidates, totalling 25 fits
[CV] END max_depth=None, max_features=log2, min_samples_split=2, n_estimators=500; total time= 1.3min
[CV] END max_depth=None, max_features=log2, min_samples_split=2, n_estimators=500; total time= 1.2min
